In [11]:
import math
import random

def ant_colony_optimization_tsp(cities, num_ants, max_iterations, alpha, beta, rho, Q, initial_pheromone):
    # --- Step 1: Define the Problem ---
    N = len(cities)
    distance_matrix = [[0.0] * N for _ in range(N)]
    heuristic_matrix = [[0.0] * N for _ in range(N)]

    for i in range(N):
        for j in range(N):
            if i != j:
                # Euclidean distance calculation
                dx = cities[i][0] - cities[j][0]
                dy = cities[i][1] - cities[j][1]
                dist = math.hypot(dx, dy)
                distance_matrix[i][j] = dist
                heuristic_matrix[i][j] = 1.0 / dist

    # --- Step 2: Initialize Parameters ---
    pheromone_matrix = [[initial_pheromone] * N for _ in range(N)]
    best_tour = None
    best_distance = float('inf')

    # --- Step 5: Iterate ---
    for iteration in range(max_iterations):
        all_tours = []
        all_distances = []

        # --- Step 3: Construct Solutions ---
        for ant in range(num_ants):
            current_city = random.randint(0, N - 1)
            visited = {current_city}
            tour = [current_city]
            tour_distance = 0.0

            while len(visited) < N:
                unvisited = [city for city in range(N) if city not in visited]
                probabilities = []

                # Calculate transition probabilities
                for next_city in unvisited:
                    tau = pheromone_matrix[current_city][next_city] ** alpha
                    eta = heuristic_matrix[current_city][next_city] ** beta
                    probabilities.append(tau * eta)

                # Normalize probabilities so they sum to 1
                total_prob = sum(probabilities)
                probabilities = [p / total_prob for p in probabilities]

                # Choose next city using weighted probabilities
                selected_city = random.choices(unvisited, weights=probabilities, k=1)[0]

                visited.add(selected_city)
                tour.append(selected_city)
                tour_distance += distance_matrix[current_city][selected_city]
                current_city = selected_city

            # Return to origin city
            origin_city = tour[0]
            tour_distance += distance_matrix[current_city][origin_city]
            tour.append(origin_city)

            all_tours.append(tour)
            all_distances.append(tour_distance)

            # Track global best
            if tour_distance < best_distance:
                best_distance = tour_distance
                best_tour = tour

        # --- Step 4: Update Pheromones ---
        # 4a. Evaporation
        for i in range(N):
            for j in range(N):
                pheromone_matrix[i][j] *= (1.0 - rho)

        # 4b. Deposit
        for k in range(num_ants):
            deposit_amount = Q / all_distances[k]
            for step in range(N):
                city_a = all_tours[k][step]
                city_b = all_tours[k][step + 1]
                pheromone_matrix[city_a][city_b] += deposit_amount
                pheromone_matrix[city_b][city_a] += deposit_amount

    # --- Step 6: Output the Best Solution ---
    return best_tour, best_distance


# ==========================================
# DRIVER CODE (Run the Algorithm)
# ==========================================
if __name__ == "__main__":
    # List of (x, y) coordinates for 8 cities
    city_coords = [
        (0, 0), (1, 5), (4, 2), (6, 6),
        (8, 3), (3, 7), (7, 1), (2, 3)
    ]

    best_route, min_dist = ant_colony_optimization_tsp(
        cities=city_coords,
        num_ants=15,
        max_iterations=100,
        alpha=1.0,
        beta=2.0,
        rho=0.5,
        Q=100.0,
        initial_pheromone=0.1
    )

    print("Best Tour (City Order):", best_route)
    print(f"Shortest Distance: {min_dist:.2f}")


Best Tour (City Order): [2, 0, 7, 1, 5, 3, 4, 6, 2]
Shortest Distance: 25.31
